In [ ]:
"""
# Transformer 30分钟K线 —— 加载本地模型并分块推理

本 notebook 只负责推理：加载 `transformer_train.py` 生成的
`transformer_model.json`，在平台注入的测试区间生成每日中证1000股票分数。

默认使用 `bigalpha_2026_stock_bar30m`。相对 1 分钟，30 分钟对「下一日收益」IC
噪声更小、内存更省。提交前请先删除旧的 1 分钟模型，再运行
`python transformer_train.py` 生成匹配的 30 分钟模型。
"""

import os

import numpy as np
import pandas as pd
import dai
import torch
import structlog

from transformer_train import (
    MODEL_PATH, BATCH, BAR_FREQ, INFER_INSTRUMENT_CHUNK, INFER_CALENDAR_DAY_CHUNK,
    StockTransformer, build_dataset, pool, train_and_save, load_model, resolve_table,
)

logger = structlog.get_logger()


def _date_blocks(start_date, end_date):
    """把测试区间切成互不重叠的小块，控制峰值内存。"""
    cursor = pd.Timestamp(start_date).normalize()
    final_day = pd.Timestamp(end_date).normalize()
    while cursor <= final_day:
        block_end = min(
            cursor + pd.Timedelta(days=INFER_CALENDAR_DAY_CHUNK - 1),
            final_day,
        )
        yield cursor.strftime("%Y-%m-%d"), block_end.strftime("%Y-%m-%d 23:59:59")
        cursor = block_end + pd.Timedelta(days=1)


def _predict_array(model, X, device):
    predictions = []
    with torch.no_grad():
        total_batches = (len(X) + BATCH - 1) // BATCH
        for batch_no, i in enumerate(range(0, len(X), BATCH), 1):
            xb = torch.from_numpy(X[i:i + BATCH]).to(device)
            predictions.append(model(xb).cpu().numpy())
            if batch_no % 100 == 0 or batch_no == total_batches:
                logger.info("推理批次进度", batch=batch_no, total_batches=total_batches)
    return np.concatenate(predictions).astype(np.float64)


def main(datasources, start_date, end_date):
    """加载训练好的 30 分钟模型，输出 ['date', 'instrument', 'score']。"""
    table = resolve_table(datasources)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    logger.info("30分钟推理任务启动", table=table, start=str(start_date), end=str(end_date),
                device=str(device), model_path=MODEL_PATH)

    if not os.path.exists(MODEL_PATH):
        raise FileNotFoundError(
            f"未找到 {MODEL_PATH}；请先运行 python transformer_train.py 重新训练。"
        )

    ckpt = load_model(MODEL_PATH, map_location=device)
    if ckpt.get("bar_frequency") != BAR_FREQ:
        raise RuntimeError(
            f"模型频率是 {ckpt.get('bar_frequency')}, 当前脚本期望 {BAR_FREQ}; "
            "请删除旧 transformer_model.json 后重新训练。"
        )
    stats = (
        np.asarray(ckpt["mean"], np.float32),
        np.asarray(ckpt["std"], np.float32),
    )
    model = StockTransformer(**ckpt["model_cfg"]).to(device)
    model.load_state_dict(ckpt["state_dict"])
    model.eval()
    logger.info("模型已加载", parameters=sum(p.numel() for p in model.parameters()),
                seq_len=ckpt["seq_len"], features=len(ckpt["feature_cols"]))

    # 截面 z-score 必须在「当日完整股票池」上做; 只按日期分块, 不再切股票子集。
    instruments = pool(start_date, end_date)
    prediction_frames = []
    block_no = 0
    for block_start, block_end in _date_blocks(start_date, end_date):
        block_no += 1
        logger.info("构建30分钟推理分块", block=block_no, start=block_start,
                    end=block_end, instruments=len(instruments))
        try:
            Xte, _, idx_df, _ = build_dataset(
                table, block_start, block_end, "infer", instruments, stats,
                cs_zscore=True,
            )
        except RuntimeError as exc:
            if "无样本" in str(exc):
                logger.warning("跳过无样本分块", block=block_no)
                continue
            raise
        idx_df["score"] = _predict_array(model, Xte, device)
        prediction_frames.append(idx_df)
        del Xte
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    if not prediction_frames:
        raise RuntimeError("测试区间没有可预测样本")
    scores = pd.concat(prediction_frames, ignore_index=True)

    stk = dai.query(
        "SELECT date, instrument FROM bigalpha_2026_instruments",
        filters={"date": [start_date, end_date]},
        compression=True,
    ).df()
    stk["date"] = pd.to_datetime(stk["date"]).dt.normalize()
    scores["date"] = pd.to_datetime(scores["date"]).dt.normalize()
    stk = stk.drop_duplicates(["date", "instrument"])
    result = (
        pd.merge(scores, stk, on=["date", "instrument"], how="inner")
        .replace([np.inf, -np.inf], np.nan)
        .dropna(subset=["score"])
        .drop_duplicates(["date", "instrument"])
        [["date", "instrument", "score"]]
        .reset_index(drop=True)
    )

    # 排名类评分只关心每日横截面顺序。转成稳定的 [-0.5, 0.5] 百分位，
    # 避免不同日期的模型输出尺度漂移影响风格剔除和回测。
    result["score"] = (
        result.groupby("date")["score"].rank(pct=True, method="average") - 0.5
    )

    expected_days = set(stk["date"])
    actual_days = set(result["date"])
    if actual_days != expected_days:
        raise RuntimeError(f"missing trading days: {sorted(expected_days - actual_days)[:10]}")
    daily_all = stk.groupby("date").size()
    daily_scored = result.groupby("date").size()
    coverage = daily_scored / daily_all
    logger.info("输出覆盖率检查", min_coverage=round(float(coverage.min()), 4),
                mean_coverage=round(float(coverage.mean()), 4), days=len(coverage))
    if (coverage < 0.60).any():
        raise RuntimeError(
            f"daily coverage below 60%: {coverage[coverage < 0.60].head().to_dict()}"
        )
    return result[["date", "instrument", "score"]]


if __name__ == "__main__":
    from bigmodule import M

    datasources = {"bar30m": "bigalpha_2026_stock_bar30m"}
    if not os.path.exists(MODEL_PATH):
        logger.info("未发现30分钟模型，开始训练", path=MODEL_PATH)
        train_and_save(datasources)

    start_date, end_date = "2024-01-01 00:00:00", "2024-12-31 23:59:59"
    score_data = main(datasources, start_date, end_date)
    print(score_data.head())
    result = M.bigalpha_eval._latest(factor_data=score_data, show=True)
